In [ ]:
#|default_exp init

# Load deps

In [ ]:
# !pip install -q torcheval
# # or
# !uv add torcheval

In [ ]:
# # if src modules imported
# from google.colab import drive
# drive.mount('/content/drive')
# import sys
# from pathlib import Path
# app_path = '/content/drive/MyDrive/Projects/miniSD'
# sys.path.append(app_path)

In [ ]:
#|export
import torch, sys,gc,traceback
import matplotlib.pyplot as plt
from math import sqrt
from functools import partial
import torchvision.transforms.functional as TF
import torch.nn.functional as F
from torch import tensor,nn
from torch.nn import init
from torcheval.metrics import MulticlassAccuracy
from datasets import load_dataset

from src.utils import store_attr
from src.learner import Callback
from src.activations import Hook, to_cpu
from src.conv import def_device

In [ ]:
from src.utils import set_seed
from src.datasets import inplace, DataLoaders
from src.conv import conv
from src.learner import MomentumLearner, DeviceCB, MetricsCB, ProgressCB
from src.activations import ActivationStats

In [ ]:
def get_model():
    return nn.Sequential(
        conv(1 ,8),
        conv(8 ,16),
        conv(16,32),
        conv(32,64),
        conv(64,10, act=False),
        nn.Flatten()
    ).to(def_device)

# Config

In [ ]:
torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)
dataset_name = "fashion_mnist"
in_lab,out_lab = 'image','label'
set_seed(42)
bs = 1024
num_dl_workers = 0

# Load data

In [ ]:
@inplace
def transformi(b): b[in_lab] = [TF.to_tensor(o).to(def_device) for o in b[in_lab]]

dsd = load_dataset(dataset_name)
tds = dsd.with_transform(transformi)
dls = DataLoaders.from_dd(
    tds, bs, num_workers=num_dl_workers
)
dt = dls.train

## Model DAG dimensions

In [ ]:
#|export --> conv
def show_model_dag(model, xb):
  print(f"input shape: {xb.shape}")
  print("===========")
  with torch.no_grad():
    for i, l in enumerate(model):
      xb = l(xb)
      print(f"layer {i}|[{type(l).__name__}] output shape: {xb.shape}")
      print("===========")

In [ ]:
xb,yb = next(iter(dt))
print(xb.shape,yb[:10])
print("===========")
mdl = get_model()
show_model_dag(mdl, xb)

# Initial train

In [ ]:
mlr = MomentumLearner(get_model(), dls, F.cross_entropy, 0.01, cbs=[DeviceCB()])
mlr.lr_find(gamma=1.1, start_lr=1e-2)
# different runs yield different results.
# TODO: how to make this more stable??

In [ ]:
metrics = MetricsCB(accuracy=MulticlassAccuracy())
astats = ActivationStats(lambda mod: isinstance(mod, nn.ReLU))
cbs = [DeviceCB(), metrics, ProgressCB(plot=True), astats]
learn = MomentumLearner(get_model(), dls, F.cross_entropy, lr=0.2, cbs=cbs)

In [ ]:
learn.fit(1)

In [ ]:
# astats.color_dim()
# astats.plot_stats()

- One obvious problem is that evan at the beginning we don't have normally (`N(0,1)`) dsitributed activations.

# Clean up GPU memory

In [ ]:
#|export
def clean_ipython_hist():
    # Code in this function mainly copied from IPython source
    if not 'get_ipython' in globals(): return
    ip = get_ipython()
    user_ns = ip.user_ns
    ip.displayhook.flush()
    pc = ip.displayhook.prompt_count + 1
    for n in range(1, pc): user_ns.pop('_i'+repr(n),None)
    user_ns.update(dict(_i='',_ii='',_iii=''))
    hm = ip.history_manager
    hm.input_hist_parsed[:] = [''] * pc
    hm.input_hist_raw[:] = [''] * pc
    hm._i = hm._ii = hm._iii = hm._i00 =  ''

def clean_tb():
    # h/t Piotr Czapla
    if hasattr(sys, 'last_traceback'):
        traceback.clear_frames(sys.last_traceback)
        delattr(sys, 'last_traceback')
    if hasattr(sys, 'last_type'): delattr(sys, 'last_type')
    if hasattr(sys, 'last_value'): delattr(sys, 'last_value')

def clean_mem():
    clean_tb()
    clean_ipython_hist()
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
clean_mem()

# Glorot/Xavier init

In [ ]:
x = torch.randn(200, 100)
for i in range(50): x = x @ torch.randn(100,100)
x[0:5,0:5]

- The result is `nan`s everywhere.
- So maybe the scale of our matrix was too big, and we need to have smaller weights?
- But if we use too small weights, we will have the opposite problem—the scale of our activations will go from 1 to 0.1
  - and after 50 layers we'll be left with zeros everywhere

In [ ]:
x = torch.randn(200, 100)
for i in range(50): x = x @ (torch.randn(100,100) * 0.01)
x[0:5,0:5]

- So we have to scale our weight matrices exactly right so that the standard deviation of our activations stays at 1.
- We can compute the exact value to use mathematically
  - as illustrated by Xavier Glorot and Yoshua Bengio in ["Understanding the Difficulty of Training Deep Feedforward Neural Networks"](http://proceedings.mlr.press/v9/glorot10a/glorot10a.pdf).
- The right scale for a given layer is $1/\sqrt{n_{in}}$, where $n_{in}$ represents the number of inputs.

In [ ]:
x = torch.randn(200, 100)
for i in range(50): x = x @ (torch.randn(100,100) * 0.1)
x[0:5,0:5]

## Background

### Variance and standard deviation

In [ ]:
t = torch.tensor([1.,2.,4.,18])
m = t.mean()
print(f"mean: {m}")
rmse = (t-m).pow(2).mean().sqrt()
mae = (t-m).abs().mean()
print(f"root mean squared error: {rmse} | mean absolute error: {mae}")
print(f"{(t-m).pow(2).mean()} | {(t*t).mean() - (m*m)}")

- RMSE and MAE are different. Why?
  - Note that we have one outlier (`18`). In the version where we square everything, it makes that much bigger than everything else.

- `(t-m).pow(2).mean()` is refered to as **variance**. It's a measure of how spread out the data is, and is particularly sensitive to outliers.

- When we take the sqrt of the variance, we get the **standard deviation**. Since it's on the same kind of scale as the original data, it's generally more interpretable. However, since `sqrt(1)==1`, it doesn't much matter which we use when talking about *unit variance* for initializing neural nets.

- The standard deviation represents if the data stays close to the mean or on the contrary gets values that are far away. It's computed by the following formula:

$$\sigma = \sqrt{\frac{1}{n}\left[(x_{0}-m)^{2} + (x_{1}-m)^{2} + \cdots + (x_{n-1}-m)^{2}\right]}$$

- where m is the mean and $\sigma$ (the greek letter sigma) is the standard deviation. Here we have a mean of 0, so it's just the square root of the mean of x squared.

- `(t-m).abs().mean()` is referred to as the **mean absolute deviation/error**. It isn't used nearly as much as it deserves to be, because mathematicians don't like how awkward it is to work with. But that shouldn't stop us, because we have computers and stuff.
- Here's a useful thing to note about variance: `(t-m).pow(2).mean()` is equal to `(t*t).mean() - (m*m)`
- Let's go steal the LaTeX from [Wikipedia](https://en.wikipedia.org/wiki/Variance):
$$\operatorname{E}\left[X^2 \right] - \operatorname{E}[X]^2$$

- What's important here is that the latter is generally much easier to work with. In particular, you only have to track two things: the sum of the data, and the sum of squares of the data. Whereas in the first form you actually have to go thru all the data twice (once to calculate the mean, once to calculate the differences).

### Covariance

$$\operatorname{cov}(X,Y) = \operatorname{E}{\big[(X - \operatorname{E}[X])(Y - \operatorname{E}[Y])\big]}$$

- It's generally more conveniently defined like so:
$$\operatorname{E}\left[X Y\right] - \operatorname{E}\left[X\right] \operatorname{E}\left[Y\right]$$

- Finally, here is the Pearson correlation coefficient. It's just a scaled (unitless) version of the same thing.
$$\rho_{X,Y}= \frac{\operatorname{cov}(X,Y)}{\sigma_X \sigma_Y}$$

In [ ]:
# `u` is twice `t`, plus a bit of randomness
u = t*2
u *= torch.randn_like(t)/10+0.95
cov = ((t-t.mean())*(u-u.mean())).mean()
corr = cov / (t.std() * u.std())
plt.scatter(t, u, label=f"u vs t | cov = {cov:.3f} | corr = {corr:.3f}");

v = torch.randn_like(t)
cov = ((t-t.mean())*(v-v.mean())).mean()
corr = cov / (t.std() * v.std())
plt.scatter(t, v, label=f"v vs t | cov = {cov:.3f} | corr = {corr:.3f}");

plt.legend();

## Xavier init derivation

- At the very beginning, our `x` vector has a mean of roughly 0. and a standard deviation of roughly 1.
  - (since we picked it that way).
- When we do `y = a @ x`, the coefficients of `y` are defined by

$$y_{i} = a_{i,0} x_{0} + a_{i,1} x_{1} + \cdots + a_{i,n-1} x_{n-1} = \sum_{k=0}^{n-1} a_{i,k} x_{k}$$

- If we go back to `y = a @ x` and assume that we chose weights for `a` that also have a mean of 0, what is the standard deviation of $y_{i}$?
- When you compute $y_{i}$, you sum $n_{in}$ products (currently 100) of one element of `a` by one element of `x`.
- So what's the mean and the standard deviation of such a product?
  - We can show mathematically that as long as the elements in `a` and the elements in `x` are independent, the mean is 0 and the std is 1.
- So $y_{i}$ ~ $N(0, \sqrt{n_{in}}))$
- This can also be seen experimentally. (see the code below)
- Hence, `math.sqrt(n_in)` being our magic number. If we scale the weights of the matrix and divide them by this `math.sqrt(n_in)`, it will give us a `y` of scale 1, and repeating the product has many times as we want won't overflow or vanish.

In [ ]:
def test_glorot(n_in, n_iter):
  mean,sqr = 0.,0.
  print(f"n_in: {n_in}")
  for i in range(n_iter):
      x = torch.randn(n_in) # ~N(0,1)
      a = torch.randn(512, n_in) # ~N(0,1)
      y = a @ x # y_i ~ N(0, 10),
      # (y_i)^2 ~ 100 * chi-2(1)
      # E((y_i)^2) = 100 | Var((y_i)^2) = 20000
      mean += y.mean().item()
      # y.mean ~ N(0, 10/512)
      sqr  += y.pow(2).mean().item()
      # y.pow(2).mean ~ 100/512 * [chi-2(512)]
      # | E=100 | Var = (100/512)^2 * (2*512) = 20000/512
  print(mean/n_iter,sqr/n_iter)
  print("============")

test_glorot(n_in = 10, n_iter= 1000)
test_glorot(n_in = 100, n_iter= 1000)

# Kaiming/He init
("He" is a Chinese surname and is pronouced like "Her", not like "Hee".)

In [ ]:
x = torch.randn(200, 100)
y = torch.randn(200)
w1 = torch.randn(100,50) / sqrt(100)
b1 = torch.zeros(50)
w2 = torch.randn(50,1) / sqrt(50)
b2 = torch.zeros(1)
def lin(x, w, b): return x @ w + b
l1 = lin(x, w1, b1)
print(f"layer 1: {l1.mean(),l1.std()}")
def relu(x): return x.clamp_min(0.)
l2 = relu(l1)
print(f"layer 1 activation: {l2.mean(),l2.std()}")

In [ ]:
x = torch.randn(200, 100)
for i in range(50):
  x = relu(x @ (torch.randn(100,100) * 0.1))
x[0:5,0:5]

In ["Delving Deep into Rectifiers: Surpassing Human-Level Performance"](https://arxiv.org/abs/1502.01852) Kaiming He et al. show that we should use the following scale instead: $\sqrt{2 / n_{in}}$, where $n_{in}$ is the number of inputs of our model.

In [ ]:
x = torch.randn(200, 100)
for i in range(50): x = relu(x @ (torch.randn(100,100) * sqrt(2/100)))
x[0:5,0:5]

# Applying an init function

In [ ]:
model = get_model()
model.apply(lambda m: print(type(m).__name__));
# applies to all modules recursively (traverses all depths)

In [ ]:
def init_weights(m):
    if isinstance(m, (nn.Conv1d,nn.Conv2d,nn.Conv3d)):
      init.kaiming_normal_(m.weight)
      # methods ending with `_` are inplace methods.
model.apply(init_weights);

In [ ]:
mlr = MomentumLearner(
    model, dls, F.cross_entropy,
    lr = 0.01, cbs=[DeviceCB()]
)
mlr.lr_find()

In [ ]:
set_seed(42)
learn = MomentumLearner(get_model().apply(init_weights), dls, F.cross_entropy, lr=0.2, cbs=cbs)
learn.fit(3)

In [ ]:
astats.color_dim()

In [ ]:
astats.plot_stats()

- Still doesn't learn
- Since we forgot to normalize the input as well

# Adding Input normalization

In [ ]:
xb,yb = next(iter(dt))
xmean,xstd = xb.mean(),xb.std()
print(xmean,xstd)

In [ ]:
#| export
class BatchTransformCB(Callback):
    def __init__(self, tfm, on_train=True, on_val=True):
      store_attr()

    def before_batch(self, learn):
        if (self.on_train and learn.training) or (self.on_val and not learn.training):
            learn.batch = self.tfm(learn.batch)

In [ ]:
def _norm(b): return (b[0]-b[0].mean())/b[0].std(),b[1]
norm = BatchTransformCB(_norm)

In [ ]:
set_seed(42)
learn = MomentumLearner(
    get_model().apply(init_weights),
    dls, F.cross_entropy, lr=0.2,
    cbs=cbs+[norm]
)
learn.fit(3)

In [ ]:
astats.color_dim()

In [ ]:
astats.plot_stats()

- It traines well
- Still, we don't get $N(0,1)$ activations
  - we're plotting the `relu` activations
  - the `relu` transformation, by definition
    - incrases the mean
    - and decreases the varinace

# General ReLU

In [ ]:
#| export
class GeneralRelu(nn.Module):
    def __init__(self, leak=None, sub=None, maxv=None):
        super().__init__()
        self.leak,self.sub,self.maxv = leak,sub,maxv

    def forward(self, x):
        x = F.leaky_relu(x,self.leak) if self.leak is not None else F.relu(x)
        if self.sub is not None: x -= self.sub
        if self.maxv is not None: x.clamp_max_(self.maxv)
        return x

def plot_func(f, start=-5., end=5., steps=100):
    x = torch.linspace(start, end, steps)
    plt.plot(x, f(x))
    plt.grid(True, which='both', ls='--')
    plt.axhline(y=0, color='k', linewidth=0.7)
    plt.axvline(x=0, color='k', linewidth=0.7)

In [ ]:
plot_func(GeneralRelu(leak=0.1, sub=0.4))
# this combination of leak and sub will gove us zero mean which is desired.
# TODO: how to find this combination automatically??
# TODO: what about the std = 1??

In [ ]:
def conv(ni, nf, ks=3, stride=2, act=nn.ReLU):
    res = nn.Conv2d(ni, nf, stride=stride, kernel_size=ks, padding=ks//2)
    if act: res = nn.Sequential(res, act())
    return res

def get_model(act=nn.ReLU, nfs = [1,8,16,32,64]):
    layers = [conv(nfs[i], nfs[i+1], act=act) for i in range(len(nfs)-1)]
    return nn.Sequential(*layers, conv(nfs[-1],10, act=None), nn.Flatten()).to(def_device)

In [ ]:
#| export
def init_weights(m, leaky=0.):
    if isinstance(m, (nn.Conv1d,nn.Conv2d,nn.Conv3d)):
      # by default, kaiming is only applied to layers with ReLU activation
      # this is how we make it applied on leaky ReLUs
      init.kaiming_normal_(m.weight, a=leaky)

In [ ]:
act_gr = partial(GeneralRelu, leak=0.1, sub=0.4)
astats = ActivationStats(lambda mod: isinstance(mod, GeneralRelu))
cbs = [DeviceCB(), metrics, ProgressCB(plot=True), astats, norm]
iw = partial(init_weights, leaky=0.1)
model = get_model(act_gr).apply(iw)

In [ ]:
set_seed(42)
learn = MomentumLearner(model, dls, F.cross_entropy, lr=0.2, cbs=cbs)
learn.fit(3)

In [ ]:
# astats.color_dim()

In [ ]:
# astats.plot_stats()

In [ ]:
# astats.dead_chart()

# LSUV

[All You Need is a Good Init](https://arxiv.org/pdf/1511.06422.pdf) introduces Layer-wise Sequential Unit-Variance (LSUV).

In [ ]:
#| export
# TODO: implement LSUV as a before_fit callback
# Note: it should run once not before every epoch!!
def _lsuv_stats(hook, mod, inp, outp):
    acts = to_cpu(outp)
    hook.mean = acts.mean()
    hook.std = acts.std()

def lsuv_init(model, m, m_in, xb):
    h = Hook(m, _lsuv_stats)
    count = 0
    with torch.no_grad():
        while model(xb) is not None and (abs(h.std-1)>1e-3 or abs(h.mean)>1e-3):
            m_in.bias -= h.mean
            m_in.weight.data /= h.std
            count += 1
        print(f"layer: {type(m_in).__name__} | mean={h.mean} | std={h.std} | iterations={count}")
    h.remove()

In [ ]:
model = get_model(act_gr)
relus = [o for o in model.modules() if isinstance(o, GeneralRelu)]
convs = [o for o in model.modules() if isinstance(o, nn.Conv2d)]
# for ms in zip(relus,convs): print(ms)
nxb = (xb - xb.mean()) / xb.std()
# since we use BN callback, we have to normalize first batch to apply LSUV
for ms in zip(relus,convs): lsuv_init(model, *ms, nxb)

In [ ]:
set_seed(42)
learn = MomentumLearner(model, dls, F.cross_entropy, lr=0.2, cbs=cbs)
learn.fit(3)

In [ ]:
# astats.plot_stats()

# Batch Normalization


Sergey Ioffe and Christian Szegedy released ["Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift"](https://arxiv.org/abs/1502.03167) in 2015, saying:

> Training Deep Neural Networks is complicated by the fact that the distribution of each layer's inputs changes during training, as the parameters of the previous layers change. This slows down the training by requiring lower learning rates and careful parameter initialization... We refer to this phenomenon as internal covariate shift, and address the problem by normalizing layer inputs.

Their proposal is:

> Making normalization a part of the model architecture and performing the normalization for each training mini-batch. Batch Normalization allows us to use much higher learning rates and be less careful about initialization.

### LayerNorm
We'll start with [layer normalization](https://arxiv.org/abs/1607.06450), a simpler technique.

In [ ]:
class LayerNorm(nn.Module):
  def __init__(self, dummy, eps=1e-5):
    super().__init__()
    self.eps = eps
    # trainable params
    self.mult = nn.Parameter(tensor(1.))
    self.add  = nn.Parameter(tensor(0.))

  def forward(self, x):
    # Normalizing per records. NHCW --> N
    m = x.mean((1,2,3), keepdim=True)
    v = x.var ((1,2,3), keepdim=True)
    x = (x-m) / ((v+self.eps).sqrt())
    return x*self.mult + self.add

In [ ]:
#|export
def conv(ni, nf, ks=3, stride=2, act=nn.ReLU, norm=None, bias=None):
    if bias is None:
      # with BN we don't need bias.
      # for LN we do. nn.conv bias is per channel. LN bias terms is per record.
      bias = not isinstance(norm, (nn.BatchNorm1d,nn.BatchNorm2d,nn.BatchNorm3d))
    layers = [nn.Conv2d(ni, nf, stride=stride, kernel_size=ks, padding=ks//2, bias=bias)]
    if norm: layers.append(norm(nf))
    if act: layers.append(act())
    return nn.Sequential(*layers)

def get_model(act=nn.ReLU, nfs = [1,8,16,32,64], norm=None):
    layers = [conv(nfs[i], nfs[i+1], act=act, norm=norm) for i in range(len(nfs)-1)]
    return nn.Sequential(
        *layers,
        conv(nfs[-1],10, act=None, norm=False, bias=True),
        nn.Flatten()
    ).to(def_device)

In [ ]:
set_seed(42)
model = get_model(act_gr, norm=LayerNorm).apply(iw)
# model.apply(lambda m: print(type(m).__name__));
# print("=================")
# show_model_dag(model, xb)

In [ ]:
learn = MomentumLearner(model, dls, F.cross_entropy, lr=0.2, cbs=cbs)
learn.fit(3)

### BatchNorm

In [ ]:
class BatchNorm(nn.Module):
    def __init__(self, nf, mom=0.1, eps=1e-5):
        super().__init__()
        # NB: pytorch bn mom is opposite of what you'd expect
        self.mom,self.eps = mom,eps
        self.mults = nn.Parameter(torch.ones (nf,1,1))
        self.adds  = nn.Parameter(torch.zeros(nf,1,1))
        self.register_buffer('vars',  torch.ones(1,nf,1,1))
        self.register_buffer('means', torch.zeros(1,nf,1,1))

    def update_stats(self, x):
        m = x.mean((0,2,3), keepdim=True)
        v = x.var ((0,2,3), keepdim=True)
        self.means.lerp_(m, self.mom)
        self.vars.lerp_ (v, self.mom)
        return m,v

    def forward(self, x):
        if self.training:
            with torch.no_grad(): m,v = self.update_stats(x)
        else: m,v = self.means,self.vars
        x = (x-m) / (v+self.eps).sqrt()
        return x*self.mults + self.adds

In [ ]:
model = get_model(act_gr, norm=BatchNorm).apply(iw)
set_seed(42)
learn = MomentumLearner(
    model, dls, F.cross_entropy,
    lr=0.4, # the first time we are able to increase the learning rate
    cbs=cbs
)
learn.fit(3)

# Towards 90% acc on fashion MNIST

## Main Train Loop

- smaller mini batch size
- lower main `lr` to be compatible with smaller batches

In [ ]:
dls = DataLoaders.from_dd(tds, 256, num_workers=num_dl_workers)
set_seed(42)
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = MomentumLearner(model, dls, F.cross_entropy, lr=0.2, cbs=cbs)
learn.fit(3)

## Fine-tune loop with smaller `lr`

In [ ]:
learn = MomentumLearner(model, dls, F.cross_entropy, lr=0.05, cbs=cbs)
learn.fit(2)